[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_6_8/07_tag_6_8_cnn_einfach_bauen_trainieren.ipynb)

# Tag 6-8 - 07 Ein CNN einfach bauen und trainieren

Ein kleines CNN wird in zwei gleichwertigen Varianten definiert: direkt mit `nn.Sequential` und als eigene Klasse. Beide werden trainiert und danach visuell ausgewertet: Beispielbilder, Vorhersagen, Fehler und Feature Maps.


In [ ]:
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = False

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, Dataset, random_split

torch.manual_seed(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("CUDA verfügbar:", torch.cuda.is_available())
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay

def accuracy_from_logits(logits, y):
    return (logits.argmax(dim=1) == y).float().mean().item()


def train_step(model, xb, yb, loss_fn, optimizer):
    model.train()
    xb, yb = xb.to(device), yb.to(device)
    optimizer.zero_grad()
    logits = model(xb)
    loss = loss_fn(logits, yb)
    loss.backward()
    optimizer.step()
    return loss.item(), accuracy_from_logits(logits.detach(), yb)


def evaluate(model, loader, loss_fn=None):
    model.eval()
    total_loss = 0.0
    correct = 0
    seen = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            if loss_fn is not None:
                total_loss += loss_fn(logits, yb).item() * len(xb)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            seen += len(xb)
    avg_loss = total_loss / seen if loss_fn is not None else np.nan
    return avg_loss, correct / seen


def train_epochs(model, train_loader, valid_loader, loss_fn, optimizer, epochs=5):
    history = []
    for epoch in range(epochs):
        losses = []
        accs = []
        for xb, yb in train_loader:
            loss, acc = train_step(model, xb, yb, loss_fn, optimizer)
            losses.append(loss)
            accs.append(acc)
        valid_loss, valid_acc = evaluate(model, valid_loader, loss_fn)
        row = {
            "epoch": epoch + 1,
            "train_loss": float(np.mean(losses)),
            "train_acc": float(np.mean(accs)),
            "valid_loss": float(valid_loss),
            "valid_acc": float(valid_acc),
        }
        history.append(row)
        print(
            f"Epoch {row['epoch']:02d}: "
            f"train_loss={row['train_loss']:.3f}, train_acc={row['train_acc']:.3f}, "
            f"valid_loss={row['valid_loss']:.3f}, valid_acc={row['valid_acc']:.3f}"
        )
    return history


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## Daten


In [ ]:
digits = load_digits()
X = digits.images.astype(np.float32) / 16.0
y = digits.target.astype(np.int64)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)
train_loader = DataLoader(TensorDataset(torch.tensor(X_train).unsqueeze(1), torch.tensor(y_train)), batch_size=64, shuffle=True)
valid_loader = DataLoader(TensorDataset(torch.tensor(X_valid).unsqueeze(1), torch.tensor(y_valid)), batch_size=256)

fig, axes = plt.subplots(1, 10, figsize=(10, 1.8))
for digit in range(10):
    idx = np.where(y_train == digit)[0][0]
    axes[digit].imshow(X_train[idx], cmap="gray", vmin=0, vmax=1)
    axes[digit].set_title(str(digit))
    axes[digit].axis("off")


## Variante 1 - CNN direkt mit `nn.Sequential` definieren

Für eine reine Kette von Schichten ist `nn.Sequential` die kürzeste und gut lesbare Lösung.


In [ ]:
model_sequential = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
    nn.Linear(32, 10),
).to(device)
print(model_sequential)
print("Trainierbare Parameter:", count_params(model_sequential))


### Variante 1 trainieren


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_sequential.parameters(), lr=0.01)
history_sequential = train_epochs(model_sequential, train_loader, valid_loader, loss_fn, optimizer, epochs=12)


## Variante 2 - Dasselbe CNN als Klasse

Eine eigene Klasse ist besonders interessant, sobald das Modell nicht mehr nur eine gerade Kette ist: etwa bei Verzweigungen, Skip Connections, mehreren Ausgaben oder einer speziellen Verarbeitung im `forward`-Schritt. Benannte Teile wie `features` und `classifier` lassen sich außerdem leichter inspizieren, ersetzen und wiederverwenden. Für dieses einfache CNN ist `nn.Sequential` kompakter; die Klassenvariante zeigt aber die Struktur, die bei größeren Modellen flexibel bleibt.


In [ ]:
class EinfachesCNN(nn.Module):
    def __init__(self, input_channels=1, width=16, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, width, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(width, width * 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(width * 2, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model_class = EinfachesCNN().to(device)
print(model_class)
print("Trainierbare Parameter:", count_params(model_class))


### Variante 2 trainieren und vergleichen

Beide Varianten haben dieselbe Architektur und damit dieselbe Anzahl trainierbarer Parameter. Wegen zufälliger Initialisierung und der zufälligen Reihenfolge der Trainingsbatches können ihre Lernkurven trotzdem leicht voneinander abweichen.


In [ ]:
torch.manual_seed(RANDOM_STATE)
optimizer_class = torch.optim.Adam(model_class.parameters(), lr=0.01)
history_class = train_epochs(model_class, train_loader, valid_loader, loss_fn, optimizer_class, epochs=12)

plt.plot([row["valid_acc"] for row in history_sequential], marker="o", label="nn.Sequential")
plt.plot([row["valid_acc"] for row in history_class], marker="o", label="Klasse EinfachesCNN")
plt.xlabel("Epoche")
plt.ylabel("Validierungs-Accuracy")
plt.legend()
plt.show()


## Ergebnisse als Bilder


In [ ]:
model_class.eval()
xb, yb = next(iter(valid_loader))
with torch.no_grad():
    logits = model_class(xb.to(device)).cpu()
    probs = torch.softmax(logits, dim=1)
    preds = probs.argmax(dim=1)

fig, axes = plt.subplots(2, 8, figsize=(12, 4))
for ax, img, true, pred, prob in zip(axes.ravel(), xb[:16], yb[:16], preds[:16], probs.max(dim=1).values[:16]):
    ax.imshow(img.squeeze(), cmap="gray", vmin=0, vmax=1)
    color = "green" if int(true) == int(pred) else "red"
    ax.set_title(f"t:{int(true)} p:{int(pred)}\n{float(prob):.2f}", color=color)
    ax.axis("off")
plt.tight_layout()


## Confusion Matrix und Feature Maps


In [ ]:
all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in valid_loader:
        all_preds.append(model_class(xb.to(device)).argmax(dim=1).cpu())
        all_true.append(yb)
preds = torch.cat(all_preds).numpy()
true = torch.cat(all_true).numpy()
ConfusionMatrixDisplay.from_predictions(true, preds, cmap="Blues")
plt.title("Confusion Matrix (Klassenvariante)")
plt.grid(False)
plt.show()

first_conv = model_class.features[0]
first_relu = model_class.features[1]
sample = torch.tensor(X_valid[:1]).unsqueeze(1).to(device)
with torch.no_grad():
    maps = first_relu(first_conv(sample))[0, :8].cpu()
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(maps[i], cmap="gray")
    ax.set_title(f"Map {i}")
    ax.axis("off")
